# ЛР 1

## Импорты

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
from fake_useragent import UserAgent
from tqdm import tqdm
from abc import ABC, abstractmethod

## Универсальный класс парсера

In [5]:
class Parser(ABC):
    def __init__(self, min_delay: float = 1, max_delay: float = 3,
                 save_path: str = 'data.csv', save_every: int = 10):
        self.min_delay = min_delay
        self.max_delay = max_delay
        self.save_path = save_path
        self.save_every = save_every
        self.ua = UserAgent()
        self.session = requests.Session()

    def _fetch_page(self, url: str) -> str | None:
        """
        Выполняет HTTP-запрос к указанному URL.
        Возвращает текст ответа или None в случае ошибки.
        """
        headers = {'User-Agent': self.ua.random}
        try:
            response = self.session.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            return response.text
        except requests.exceptions.RequestException as e:
            print(f"Ошибка при загрузке {url}: {e}")
            return None

    @abstractmethod
    def _parse_page(self, html: str) -> list[dict]:
        """
        Принимает HTML страницы, возвращает список словарей с данными.
        """
        pass

    def parse_single(self, url: str) -> list[dict]:
        """
        Парсит одну страницу по заданному URL и возвращает извлечённые данные.
        """
        html = self._fetch_page(url)
        if html is None:
            return []
        return self._parse_page(html)

    def parse_multiple(self, urls: list[str]) -> list[dict]:
        """
        Парсит список URL, накапливает данные, применяет задержки и сохраняет промежуточные результаты.
        Информация о сохранении отображается в postfix прогресс-бара.
        """
        all_data = []
        with tqdm(total=len(urls), desc="Парсинг страниц") as pbar:
            for idx, url in enumerate(urls):
                html = self._fetch_page(url)
                if html is not None:
                    page_data = self._parse_page(html)
                    all_data.extend(page_data)

                if (idx + 1) % self.save_every == 0:
                    self._save_data(all_data)
                    pbar.set_postfix({
                        'saved': len(all_data),
                        'last_save': idx + 1
                    })

                delay = random.uniform(self.min_delay, self.max_delay)
                time.sleep(delay)
                pbar.update(1)

        if all_data:
            self._save_data(all_data)
            print(f"Парсинг завершён. Всего собрано {len(all_data)} записей.")
        return all_data

    def _save_data(self, data: list[dict]) -> None:
        """Сохраняет данные в CSV без лишнего вывода."""
        if not data:
            return
        df = pd.DataFrame(data)
        df.to_csv(self.save_path, index=False, encoding='utf-8')

    def _save_data(self, data: list[dict]) -> None:
        """
        Сохраняет список словарей в CSV-файл.
        """
        if not data:
            return
        df = pd.DataFrame(data)
        df.to_csv(self.save_path, index=False, encoding='utf-8')

    @staticmethod
    def generate_urls(base_url: str, num_pages: int,
                      page_param: str = 'page') -> list[str]:
        """
        Утилита для генерации списка URL
        """
        urls = []
        for i in range(1, num_pages + 1):
            if '{}' in base_url:
                urls.append(base_url.format(i))
            else:
                urls.append(f"{base_url}?{page_param}={i}")
        return urls

## Использование

In [7]:
class QuotesParser(Parser):
    def _parse_page(self, html: str) -> list[dict]:
        soup = BeautifulSoup(html, 'html.parser')
        quotes = []
        for quote_div in soup.find_all('div', class_='quote'):
            text = quote_div.find('span', class_='text').text
            author = quote_div.find('small', class_='author').text
            tags = [tag.text for tag in quote_div.find_all('a', class_='tag')]
            quotes.append({
                'text': text,
                'author': author,
                'tags': ', '.join(tags)
            })
        return quotes


parser = QuotesParser(min_delay=1, max_delay=3, save_path='quotes.csv', save_every=5)

urls = parser.generate_urls("http://quotes.toscrape.com/page/{}", 10)
all_data = parser.parse_multiple(urls)
print(f"Всего собрано {len(all_data)} цитат")

Парсинг страниц: 100%|████████████████████████████████████████| 10/10 [00:22<00:00,  2.30s/it, saved=100, last_save=10]

Парсинг завершён. Всего собрано 100 записей.
Всего собрано 100 цитат


In [13]:
class BooksParser(Parser):
    def _parse_page(self, html: str, page_url: str = None) -> list[dict]:
        soup = BeautifulSoup(html, 'html.parser')
        books = []
        for article in soup.find_all('article', class_='product_pod'):

            title_tag = article.h3.a
            title = title_tag.get('title', '') if title_tag else ''

            price_tag = article.find('p', class_='price_color')
            price = price_tag.text.strip() if price_tag else ''

            rating_tag = article.find('p', class_='star-rating')
            rating = ''
            if rating_tag:
                classes = rating_tag.get('class', [])
                if len(classes) > 1:
                    rating = classes[1]

            rel_link = article.div.a.get('href', '') if article.div and article.div.a else ''
            full_url = urljoin(page_url, rel_link) if page_url and rel_link else ''

            books.append({
                'title': title,
                'price': price,
                'rating': rating,
                'url': full_url
            })
        return books


parser = BooksParser(min_delay=1, max_delay=3, save_path='books.csv', save_every=5)

urls = parser.generate_urls("http://books.toscrape.com/catalogue/page-{}.html", 50)
all_books = parser.parse_multiple(urls)

print(f"Всего собрано книг: {len(all_books)}")
if all_books:
    print("\nПример первых 5 записей:")
    for book in all_books[:5]:
        print(book)

Парсинг страниц: 100%|███████████████████████████████████████| 50/50 [01:55<00:00,  2.31s/it, saved=1000, last_save=50]

Парсинг завершён. Всего собрано 1000 записей.
Всего собрано книг: 1000

Пример первых 5 записей:
{'title': 'A Light in the Attic', 'price': 'Â£51.77', 'rating': 'Three', 'url': ''}
{'title': 'Tipping the Velvet', 'price': 'Â£53.74', 'rating': 'One', 'url': ''}
{'title': 'Soumission', 'price': 'Â£50.10', 'rating': 'One', 'url': ''}
{'title': 'Sharp Objects', 'price': 'Â£47.82', 'rating': 'Four', 'url': ''}
{'title': 'Sapiens: A Brief History of Humankind', 'price': 'Â£54.23', 'rating': 'Five', 'url': ''}
